In [ ]:
# # load all annotations
# import objaverse.xl as oxl
# import pandas as pd
# import re
# import json

# all_annotations = oxl.get_annotations(
#     download_dir="~/.objaverse" # default download directory
# )

# # objaverse
# # annotations = all_annotations[all_annotations["metadata"]!="{}"]
# # annotations["description"] = annotations["metadata"].map(json.loads).map(lambda d: d.get("filename", d.get("title")).replace("_", " "))
# # os.environ["THINGIVERSE_COOKIE"] = ""
# # os.environ["THINGIVERSE_USER_AGENT"] = "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/143.0.0.0 Safari/537.36"

# # github
# annotations = all_annotations[all_annotations["source"]=="github"]
# annotations["description"] = annotations["fileIdentifier"].map(lambda d: d.split("/")[-1].replace("_", " ").split(".")[0])


In [ ]:
# source
# github         5236361
# sketchfab       796031
# smithsonian       2407
# thingiverse    3732212
# dtype: int64

# fileType
# abc         1053
# blend     238201
# dae       123438
# fbx      1509719
# glb       186686
# gltf      127287
# obj      1620083
# ply       839343
# stl       584870
# usdz        5681

In [ ]:
# # filter annotations with allowed filetype and keyword
# import os
# import shutil
# import subprocess
# from tqdm import tqdm

# # Example keyword
# keywords = ["folding chair", "briefcase", "book", "toilet", "bu", "raccoon", "backpack", "payphone", "makeup", "ladder", "sock", "telephone booth", "traffic barrel", "bag", "coin", "shuttle", "wheelbarrow", "sculpture", "plank", "case", "tote", "lighter", "jacket", "pinecone", "card", "fountain", "motorcycle", "broom", "mirror", "dog", "can", "sunglas", "bottle", "squirrel", "van", "badge", "vest", "wrapper", "planter", "sunglass", "rebar", "leave", "rabbit", "cyclist", "ambulance", "bench", "puddle", "cat", "towel", "hat", "hose", "branch", "bird", "box", "toy", "ball", "notebook", "toolbox", "chair", "bollard", "umbrella", "bowl", "traffic cone", "police", "truck", "cardboard", "tissue", "comb", "table", "mask", "trash can", "shoe", "runner", "pallet", "snow pile", "lipstick", "metal pipe", "car", "leaf", "bike", "camera", "headphone", "lottery", "shovel", "mailbox", "stroller", "dumpster", "collar", "wallet", "pen", "container", "bike rack", "parking meter", "feather", "newspaper", "skateboard", "napkin", "sign", "receipt", "charger", "bucket", "battery", "scooter", "taxi", "frisbee", "phone", "scarf", "helmet", "cup", "glove", "bicycle", "blanket", "wheelchair", "key", "recycling bin", "cable", "minivan", "bus stop", "triangle", "drive", "keychain"]
# allowed_filetype = ["blend", "fbx", "glb", "gltf", "usdz"]
# annotations_keyword = {}

# for keyword in tqdm(keywords):
#     # check if keyword is in description
#     mask = annotations["description"].apply(lambda x: " "+keyword.lower()+" " in x or x.endswith(" "+keyword.lower()) or x.startswith(keyword.lower()+" "))
#     keyword_plural = keyword + "s"
#     mask = mask | annotations["description"].apply(lambda x: " "+keyword_plural.lower()+" " in x or x.endswith(" "+keyword_plural.lower()) or x.startswith(keyword_plural.lower()+" "))
#     mask = mask & annotations["fileType"].isin(allowed_filetype)
#     annotations.loc[mask, 'keyword'] = keyword
#     # set object_name attribute to index+file name
#     annotations.loc[mask, 'object_name'] = [
#         f"{file_id.split('/')[-1].replace(' ', '_').lower().split('.')[0]}_{idx}"
#         for idx, file_id in annotations.loc[mask, 'fileIdentifier'].items()
#     ]
#     annotations_keyword[keyword] = annotations[mask]

#     print(f"Found {mask.sum()} {keyword} objects")


In [ ]:
import pandas as pd

# merge annotations_keyword and save as parquet
# keywords_df = pd.concat(annotations_keyword.values(), ignore_index=True)
# keywords_df.to_parquet("keywords_annotations.parquet")

# load keywords_annotations.parquet
keywords_df = pd.read_parquet("keywords_annotations.parquet")
annotations = keywords_df
keywords = keywords_df.keyword.unique()
annotations_keyword = {}
for keyword in keywords:
    annotations_keyword[keyword] = keywords_df[keywords_df.keyword == keyword]
    print(f"Found {len(annotations_keyword[keyword])} {keyword} objects")



In [ ]:
# download objects with file identifier
import os
import github_utils

n_download_per_keyword = 50
downloaded_objects = {}

for keyword in keywords:
    download_folder = os.path.abspath(f"./data/objects/{keyword}")
    sampled_annotations = annotations_keyword[keyword]
    sampled_annotations = sampled_annotations.sample(min(n_download_per_keyword, len(sampled_annotations)))
    file_urls = sampled_annotations.fileIdentifier.values.tolist()

    print(f"Starting batch download for {len(file_urls)} files to {download_folder}...")
    # Use multiprocessing to speed up
    results = github_utils.download_batch(file_urls, download_folder, num_processes=8)

    # Process results
    success_count = 0
    downloaded = {}

    print("\nDownload Summary:")
    for url, (success, message, local_path) in results.items():
        if success:
            success_count += 1
            downloaded[url] = local_path
        else:
            # Optional: Print failures if not already printed by github_utils
            pass
    downloaded_objects[keyword] = downloaded

    print(f"\nSuccessfully downloaded {success_count}/{len(file_urls)} files.")
    print("Batch download finished.")
for keyword in keywords:
    print(f"Downloaded {len(downloaded_objects[keyword])} {keyword} objects")

In [ ]:
# render images
import os
import multiprocessing
from functools import partial
import json
import subprocess
from tqdm import tqdm

output_base_dir = "./data/render/"
log_file_path = os.path.join(output_base_dir, "render_log.txt")

# Ensure output directory exists
os.makedirs(output_base_dir, exist_ok=True)

def process_single_object(args):
    keyword, file_identifier, local_path, output_base_dir, annotations, log_queue = args
    
    try:
        obj_name = annotations[annotations.fileIdentifier==file_identifier].object_name.values[0]
        folder_name = f"{keyword}/{obj_name}"
        object_dir = os.path.join(output_base_dir, folder_name)
        
        metadata = {
            "keyword": keyword,
            "url": file_identifier,
            "local_path": local_path,
            "filename": os.path.basename(local_path),
        }
        
        # Render images using Blender
        blender_executable = "/snap/bin/blender"
        if not os.path.exists(blender_executable):
            blender_executable = "blender"
        
        blender_cmd = [
            blender_executable,
            "--background",
            "--python",
            "render_blender.py",
            "--",
            "--input", local_path,
            "--output_dir", object_dir,
            "--metadata", json.dumps(metadata)
        ]
        
        log_queue.put(f"--- Processing {folder_name} ---")
        log_queue.put(f"{blender_cmd}")

        # Run blender, capture output
        result = subprocess.run(blender_cmd, check=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
        log_queue.put(result.stdout)
        
        return True, obj_name
        
    except subprocess.CalledProcessError as e:
        log_queue.put(f"\n[ERROR] Failed to render {folder_name}: {e}")
        if e.stdout:
            log_queue.put(e.stdout)
        return False, obj_name
    except Exception as e:
        log_queue.put(f"\n[ERROR] Unexpected error processing {folder_name}: {e}")
        return False, obj_name

def logger_process(queue, log_path):
    with open(log_path, "w") as f:
        while True:
            msg = queue.get()
            if msg == "DONE":
                break
            f.write(str(msg) + "\n")
            f.flush()

print(f"Logs will be saved to {log_file_path}")

# Setup multiprocessing manager for logging queue
manager = multiprocessing.Manager()
log_queue = manager.Queue()

# Start logger process
logger = multiprocessing.Process(target=logger_process, args=(log_queue, log_file_path))
logger.start()

tasks = []
for keyword in keywords:
    for file_identifier, local_path in downloaded_objects[keyword].items():
            tasks.append((keyword, file_identifier, local_path, output_base_dir, annotations, log_queue))

with multiprocessing.Pool(processes=32) as pool:
    results = list(tqdm(pool.imap(process_single_object, tasks), total=len(tasks), desc=f"Rendering objects"))
        
# Stop logger
log_queue.put("DONE")
logger.join()

print(f"Done processing. Data saved in {output_base_dir}. Check {log_file_path} for details.")

In [ ]:
import json
json_file = open("data/result/all_selected_objects.json", "r")
data = json.load(json_file)

given_height = {
  "vest": 0.70,
  "briefcase": 0.35,
  "hiking bag": 0.65,
  "backpack": 0.8,
  "socks": 0.25,
  "ladder": 0.15,
  "living room chair": 1.10,
  "leather chair": 1.10,
  "dinning chair": 1.00,
  "black_office_chair": 1.20,
  "dog": 0.70,
  "office phone": 0.15,
  "old telephone": 0.35,
  "cell phone": 0.05,
  "office desk": 0.80,
  "wooden table": 0.80,
  "table": 0.80,
  "coffee table": 0.55,
  "chair": 1.00,
  "office chair": 1.20,
  "recycling bin": 1.10,
  "trash can": 1.05,
  "metal trashcan": 1.05,
  "trash bag": 0.95,
  "bus_stop": 3.00,
  "bus": 3.30,
  "headphones": 0.22,
  "police hat": 0.22,
  "offroad truck": 2.10,
  "food truck": 3.40,
  "garbage truck": 3.00,
  "cat": 0.35,
  "vans shoes": 0.25,
  "white shoes": 0.25,
  "black shoes": 0.25,
  "shoe box": 0.3,
  "parking meter": 1.35,
  "puddle": 0.2,
  "Purse": 0.28,
  "camera bag": 0.35,
  "handbag": 0.28,
  "house mailbox": 1.20,
  "apartment mailbox": 1.40,
  "tissues": 0.3,
  "tissue box": 0.35,
  "wallet": 0.05,
  "bathroom mirror": 0.90,
  "lock box": 0.5,
  "clay pot": 0.45,
  "cargo pallet": 0.15,
  "earth_ball": 0.30,
  "soccer_ball": 0.30,
  "Soccer ball": 0.30,
  "soccer ball": 0.30,
  "dragon_ball": 0.15,
  "bowling ball": 0.23,
  "beach ball": 0.35,
  "bicycle": 1.15,
  "storage tote": 0.55,
  "broom": 1.45,
  "van": 2.60,
  "police van": 2.60,
  "race_car": 1.70,
  "electric car": 1.70,
  "red_scooter": 1.25,
  "white_scooter": 1.25,
  "books": 0.28,
  "single_book": 0.28,
  "skateboard": 0.15,
  "robot toy": 0.40,
  "rice bowl": 0.10,
  "boxing_gloves": 0.2,
  "glove": 0.25,
  "viking helmet": 0.5,
  "magician hat": 0.38,
  "sun hat": 0.28,
  "orange_jacket": 0.95,
  "purple_jacket": 0.95,
  "sports_jacket": 0.95,
  "dinner_jacket": 0.95,
  "garden fountain": 1.60,
  "water_fountain": 1.40,
  "taxi": 1.75,
  "bike rack": 0.4,
  "gardening hose": 0.35,
  "lilac_bike": 1.15,
  "black_bike": 1.15,
  "rubber duck": 0.10,
  "seagull": 0.45,
  "bottle with red liquid": 0.32,
  "nursing bottle": 0.12,
  "projectile_bottle": 0.32,
  "white_glue_bottle": 0.32,
  "ale_bottle": 0.32,
  "bleach_bottle": 0.35,
  "squirrel": 0.28,
  "sand bucket": 0.28,
  "strereo_camera_sensor": 0.12,
  "digital camera": 0.15,
  "steampunk_camera": 0.12,
  "camera_bag": 0.35,
  "toilet": 0.85,
  "toilet paper": 0.12,
  "park_bench": 0.5,
  "white_bench": 1.00,
  "park bench with back": 1.10,
  "umbrella": 1.10,
  "notebook": 0.30,
  "baby stroller": 1.15,
  "indian_feather_hat": 0.65,
  "fleece_blankets": 0.22,
  "warm_blanket": 0.22,
  "ambulance": 2.80,
  "shovel": 1.55,
  "beach shovel": 0.5,
  "pack of paper towels": 0.38,
  "stack of towels": 0.30,
  "paper towel": 0.28,
  "coffee cup": 0.12
}

for item in data:
    obj_name = item["custom_name"]
    if not obj_name in given_height:
        print(f"No height for {obj_name}")
        continue
    item["height"] = given_height[obj_name]

with open("data/result/all_selected_objects.json", "w") as f:
    json.dump(data, f, indent=4)
